In [1]:
 #step 1 import libraries

import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    Conv2DTranspose,
    Concatenate,
    Dropout
)

from tensorflow.keras.models import Model

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau

)
from tensorflow.keras.optimizers import Adam
import warnings

warnings.filterwarnings("ignore")

print("Tensorflow version:", tf.__version__)

print("GPU Available : ", len(tf.config.list_physical_devices('GPU')))



Tensorflow version: 2.21.0
GPU Available :  0


In [2]:
#step 2 load images and masks


IMAGE_PATH="/content/drive/MyDrive/Covid 19 DATASET file/frames"

MASK_PATH="/content/drive/MyDrive/Covid 19 DATASET file/masks"

image_files = sorted(os.listdir(IMAGE_PATH))
mask_files = sorted(os.listdir(MASK_PATH))

image_paths = [os.path.join(IMAGE_PATH, img) for img in image_files]
mask_paths = [os.path.join(MASK_PATH, msk) for msk in mask_files]

print("total images : ", len(image_paths))
print("total masks:", len(mask_paths))




total images :  2729
total masks: 2744


In [3]:
#step 3 read and preprocess

IMG_SIZE = 256

images = []
masks = []

for img_path, mask_path in zip(image_paths,mask_paths):
    image= cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    image=cv2.resize(image,(IMG_SIZE,IMG_SIZE))
    image= image / 255.0

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(
            mask,
            (IMG_SIZE,IMG_SIZE),
            interpolation = cv2.INTER_NEAREST
    )
    mask = (mask > 127).astype(np.float32)
    image = np.expand_dims(image, axis=-1)
    mask = np.expand_dims(mask, axis=-1)
    images.append(image)
    masks.append(mask)


images = np.array(images, dtype=np.float32)
masks = np.array(masks, dtype = np.float32)

print("Images  shape : ", images.shape)
print("masks shape : ", masks.shape)

print("image range")

print("minimum value : ", images.min())
print("maximum value : ", images.max())


KeyboardInterrupt: 

In [ ]:
#step 4 display iamges and masks

NUM_SAMPLES = 5

plt.figure(figsize=(12,10))

for i in range (NUM_SAMPLES):

    plt.subplot(NUM_SAMPLES,2,2*i + 1)
    plt.imshow(images[i].squeeze(), cmap="gray")
    plt.title(f"CT Images {i + 1}")
    plt.axis("off")

    plt.subplot(NUM_SAMPLES,2,2*i + 2)
    plt.imshow(masks[i].squeeze(), cmap="gray")
    plt.title(f"Mask {i + 1}")
    plt.axis("off")

plt.tight_layout()
plt.show()



In [ ]:
#cell 5
x_train, x_temp, y_train, y_temp = train_test_split (
    images,
    masks,
    test_size = 0.20,
    random_state = 42,
    shuffle = True
)

x_val, x_test, y_val, y_test = train_test_split(
    x_temp,
    y_temp,
    test_size=0.50,
)

print("Training Images  : " , len(x_train))
print("Validation Images  : " , len(x_val))
print("Test Images  : " , len(x_test))

In [ ]:
#cell 6
BATCH_SIZE = 8

train_dataset =  tf.data.Dataset.from_tensor_slices(
    (x_train,y_train)
)

train_dataset = train_dataset.shuffle(
    buffer_size = len(x_train),
    reshuffle_each_iteration = True
)

train_dataset = train_dataset.batch(BATCH_SIZE)
train_dataset = train_dataset.prefetch(
    tf.data.AUTOTUNE
)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (x_val,y_val)
)

val_dataset = val_dataset.batch(BATCH_SIZE)
val_dataset = val_dataset.prefetch(
    tf.data.AUTOTUNE
)

test_dataset = tf.data.Dataset.from_tensor_slices(
    (x_test,y_test)
)

test_dataset = test_dataset.batch(BATCH_SIZE)
test_dataset = test_dataset.prefetch(
    tf.data.AUTOTUNE
)


print("Training Batches :", len(train_dataset))
print("Validation Batches :", len(val_dataset))
print("Testing Batches :", len(test_dataset))



In [ ]:
#cell 7
def conv_block(inputs, filters):
    x= Conv2D(
        filters,
        (3,3),
        padding ="same",
        activation = "relu"
    )(inputs)

    x=Conv2D(
        filters,
        (3,3),
        padding ="same",
        activation = "relu"
    )(x)
    return x


def encoder_block(inputs, filters):
    x= conv_block(inputs, filters)
    p = MaxPooling2D((2,2))(x)

    return x,p

def decoder_block(inputs, skip_features, filters):
    x= Conv2DTranspose(
        filters,
        (2,2),
        strides = 2,
        padding = "same",

    )(inputs)
    x= Concatenate()([x,skip_features])
    x= conv_block( x, filters)

    return x



def build_unet(input_shape=(256,256,1)):
    inputs  = Input(input_shape)

    #encoder part
    s1,p1 = encoder_block(inputs, 64)
    s2,p2 = encoder_block(p1, 128)
    s3,p3 = encoder_block(p2, 256)
    s4,p4 = encoder_block(p3, 512)

    #bottlenech
    b1=conv_block(p4,1024)

    #decoder
    d1=decoder_block(b1,s4,512)
    d2=decoder_block(d1,s3,256)
    d3=decoder_block(d2,s2,128)
    d4=decoder_block(d3,s1,64)

    outputs= Conv2D(
        1,
        (1,1),
        padding="same",
        activation = "sigmoid"
    )(d4)

    model = Model(inputs, outputs)
    return model

model = build_unet()
model.summary()

In [ ]:
#cell 8
#loss functions
import tensorflow.keras.backend as k

def dice_coefficient(y_true, y_pred):
    y_true = k.flatten(y_true)
    y_pred = k.flatten(y_pred)
    intersection = k.sum(y_true*y_pred)
    dice = (2. * intersection + 1e-7)/(
        k.sum(y_true)+k.sum(y_pred)+1e-7)
    return dice


def dice_loss(y_true, y_pred):
    return 1 -dice_coefficient(y_true,y_pred)

def iou(y_true, y_pred):
    y_true = k.flatten(y_true)
    y_pred= k.flatten(y_pred)
    intersection = k.sum(y_true*y_pred)
    union = (
        k.sum(y_true)
        + k.sum(y_pred)
        -intersection
    )
    return (intersection + 1e-7)/(union + 1e-7)



bce = tf.keras.losses.BinaryCrossentropy()

def total_loss(y_true,y_pred):
    return bce(y_true, y_pred) + dice_loss(y_true, y_pred)

In [ ]:
#step 9
model.compile (
    optimizer = Adam(learning_rate = 1e-4),
    loss = total_loss,
    metrics = [
        dice_coefficient,
        iou,
        "accuracy"
    ]
)

print("Mdoel is ready")

checkpoint = ModelCheckpoint(
    filepath="best_unet_model.keras",
    monitor = "val_loss",
    save_best_only = True,
    verbose=1
)

early_stoping = EarlyStopping(
    monitor = "val_loss",
    patience =10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=  0.5,
    patience = 5,
    min_lr=1e-7,
    verbose=1
)

callbacks = [
    checkpoint,
    early_stoping,
    reduce_lr

]

print("Callbacks are ready")

In [ ]:
#step 10


EPOCH = 10

history  = model.fit(
    train_dataset,
    validation_data = val_dataset,
    epochs=EPOCH,
    callbacks=callbacks,
    verbose=1
)

print("Model training completed")

model.save("final_unet_model.keras")

print("Final Model Saved")

import pickle
with open ("training_history.pkl","wb") as file:
     pickle.dump(history.history, file)
print(" training history saved")


In [ ]:
#plot 1
plt.figure(figsize=(18,5))

plt.subplot(1,3,1)
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title(" Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

#plot 2
plt.subplot(1,3,2)
plt.plot(history.history["dice_coefficient"], label="Training Dice")
plt.plot(history.history["val_dice_coefficient"], label="Validation Dice")
plt.title("Dice Coefficient")
plt.xlabel("Epoch")
plt.ylabel("Dice Score")
plt.legend()
plt.grid(True)

#plot 3
plt.subplot(1,3,3)
plt.plot(history.history["iou"], label="Training iou")
plt.plot(history.history["val_iou"], label="Validation iou")
plt.title("IOU")
plt.xlabel("Epoch")
plt.ylabel("IOU")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
#cell 12
#test


print("testing on test dataset")

result = model.evaluate(
    test_dataset,
    verbose=1
)

print(f"Test loss : {result[0]:.4f} ")
print(f"Dice Score : {result[1]:.4f} ")
print(f"IOU Score : {result[2]:.4f} ")
print(f"Test Accuracy : {result[3]:.4f} ")



In [ ]:
NUM_IMAGES = 5

random_indices = np.random.choice(
    len(x_test),
    NUM_IMAGES,
    replace =False
)

plt.figure(figsize=(15,NUM_IMAGES*5 ))


for i, idx in enumerate(random_indices):
    image = x_test[idx]
    true_mask = y_test[idx]

    prediction = model.predict(
        np.expand_dims(image, axis=0),
        verbose=0
    )

    prediction = prediction[0]
    prediction = (prediction > 0.5).astype(np.float32)


    #image
    plt.subplot(NUM_IMAGES,3,i*3+1)
    plt.imshow(image.squeeze(), cmap="gray")
    plt.title("Original CT image")
    plt.axis("off")

    #mask original
    plt.subplot(NUM_IMAGES,3,i*3+2)
    plt.imshow(true_mask.squeeze(), cmap="gray")
    plt.title("Original Mask")
    plt.axis("off")

    #mask Predicted
    plt.subplot(NUM_IMAGES,3,i*3+3)
    plt.imshow(prediction.squeeze(), cmap="gray")
    plt.title("Predicted Mask")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
!git config --global user.name "YourGitHubUsername"
!git config --global user.email "YourGitHubEmail"

In [ ]:
!ls -la /content/

In [ ]:
!ls -la /content/COVID-19/

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!find /content/drive/MyDrive -maxdepth 3 -iname '*covid*'

In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/covid 19.ipynb" /content/COVID-19/

In [ ]:
!cp -r "/content/drive/MyDrive/Covid 19 DATASET" /content/COVID-19/